# Stage 4：Olist 訂單是否延遲預測

這本 Notebook 帶你完成「訂單核准時，預測最後是否超過預估送達日期」的流程。特徵與訓練邏輯由 `scripts/delay_features.py`、`scripts/train_delay_model.py` 共用，這裡負責導覽、檢查結果與示範推論。

請使用 **Shop (Python 3.12)** kernel。資料來自本機 `data/raw/olist/` 的六份 CSV，執行時不需要 MySQL。模型與大量逐筆預測保留在本機，Notebook 不內嵌這些檔案。


## 1. 先定義預測時點與答案

- **預測時點**：`order_approved_at`，也就是訂單／付款核准時間。
- **延遲標記**：實際送達的「日期」晚於預估送達的「日期」時，`delayed = 1`，其餘為 0。
- **答案可得時間**：`label_available_time` 是實際送達時間。訓練時必須檢查當時是否已知道答案。

例如預估 1 月 10 日送達，實際 1 月 10 日 23:59 送達仍算準時；1 月 11 日 00:00 送達才算延遲。Stage 3 用完整時間戳記比較午夜的統計方式與這裡不同，因此延遲率可能不同。

歷史訓練資料限於已送達、所需時間完整且時序合理的訂單。這個模型預測的是這群訂單的配送延遲，不包含取消或始終未送達的風險。


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import joblib
import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (cwd, cwd.parent)
     if (candidate / "scripts" / "delay_features.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("請從專案根目錄或 notebooks 資料夾執行這本 Notebook。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.delay_features import (
    CATEGORICAL_FEATURES, FEATURES, NUMERIC_FEATURES, build_delay_dataset,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "olist"
MODEL_PATH = PROJECT_ROOT / "models" / "olist_delay_model.joblib"
METRICS_PATH = PROJECT_ROOT / "reports" / "stage4" / "metrics.json"

print("專案根目錄：", PROJECT_ROOT)
print("Python：", sys.executable)


## 2. 建立每訂單一列的資料

商品明細與付款紀錄會各自彙總，再合併到訂單，避免一張訂單有多個商品與多筆付款時把金額重複加總。下面只顯示資料量、延遲比例與特徵摘要。


In [ ]:
dataset = build_delay_dataset(RAW_DIR)

display(pd.DataFrame([{
    "合格訂單數": dataset.shape[0],
    "資料欄位數": dataset.shape[1],
    "模型特徵數": len(FEATURES),
    "延遲訂單數": int(dataset["delayed"].sum()),
    "延遲比例": float(dataset["delayed"].mean()),
    "最早核准時間": dataset["prediction_time"].min(),
    "最晚核准時間": dataset["prediction_time"].max(),
}]))


## 3. 核准時可以知道哪些特徵

數值特徵包含商品數、商品與運費總額、付款資訊、商品重量與體積、預估配送天數、核准等待時間、購買時段及跨州商品比例。類別特徵包含顧客州別、主要賣家州別、主要品類與付款方式。完整欄位使用明確白名單 `FEATURES`。

實際送達、交給物流的時間、評論、最終訂單狀態，以及訂單／顧客 ID 都不進入模型。前幾項屬於預測之後才知道的資訊，ID 也不作為預測特徵。缺值補齊、標準化與類別編碼都放在 Pipeline，只用訓練集學習。

Olist 是事後快照資料，沒有每次商品、付款或預估交期變更的完整時間紀錄。本階段假設這些欄位在核准時已知；未來接真實系統時，應保存核准當下的快照來驗證這個假設。


In [ ]:
feature_summary = pd.DataFrame({
    "feature": FEATURES,
    "kind": ["numeric" if name in NUMERIC_FEATURES else "categorical"
             for name in FEATURES],
    "missing_rate": [float(dataset[name].isna().mean()) for name in FEATURES],
    "distinct_values": [dataset[name].nunique(dropna=True) for name in FEATURES],
})
display(feature_summary)


## 4. 依時間切分，排除當時還不知道的答案

訓練程式依 `prediction_time` 排序，以約 60%／20%／20% 的訂單量建立 train、validation、test；同一天的訂單全部放在同一區間，邊界日歸到後面的區間。

只做時間切分仍可能用到未來答案，因此還會做 **label maturity purge**：

- train 的送達時間必須嚴格早於 validation 開始時間。
- validation 的送達時間必須嚴格早於 test 開始時間。

因此三組實際筆數會受到日期邊界與 purge 影響。這次保留選定的原始 train 模型，不在選模後合併 validation 重訓；模型、前處理與門檻先固定，再對 test 預測一次。


## 5. 使用現有訓練成果，必要時由 CLI 建立

`RUN_TRAINING = False` 時，模型與報告都存在就直接讀取；任一檔案不存在時會自動執行訓練。若要重新產生整套成果，把設定改為 `True`。

CLI 使用目前 kernel 的 `sys.executable`，確保與 Notebook 使用同一個 Python 環境。


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING or not MODEL_PATH.is_file() or not METRICS_PATH.is_file():
    command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "train_delay_model.py"),
        "--data-dir", str(RAW_DIR),
        "--output-dir", str(METRICS_PATH.parent),
        "--model-path", str(MODEL_PATH),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("使用已存在的模型與 metrics.json。")


In [ ]:
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
artifact = joblib.load(MODEL_PATH)

if artifact["feature_names"] != list(FEATURES):
    raise ValueError("模型特徵與目前程式不同，請重新訓練。")
if artifact["selected_model"] != metrics["selected_model"]:
    raise ValueError("模型與報告不一致，請重新產生同一批訓練成果。")

split_info = metrics["metadata"]["split"]
print("Validation 開始：", split_info["validation_start"])
print("Test 開始：", split_info["test_start"])
display(pd.DataFrame.from_dict(split_info["partitions"], orient="index"))


## 6. 在 validation 選模型與決策門檻

這次比較 Dummy baseline、Logistic Regression、Random Forest。Dummy 只作為基本參考；可選模型是 Logistic Regression 與 Random Forest，分別學習線性關係與樹模型的非線性關係。

先在兩個可選模型中，以 validation 的 **Average Precision（AP）最高者選模型**，並檢查 AP 是否高於 Dummy。AP 衡量排序能力，在延遲訂單較少的資料中特別有參考價值。再只使用被選模型的 validation 分數，選擇 **F1 最高的門檻**。F1 同時考慮 precision 與 recall；這是本階段的實驗策略，未來可依人工查核量或延遲漏判成本調整。

下表比較候選模型的 validation 結果。這些分類指標使用 0.5 門檻；AP 和 ROC AUC 使用連續分數，不受分類門檻影響。


In [ ]:
validation_rows = []
for model_name, result in metrics["model_comparison"].items():
    row = {"model": model_name, "status": result["status"]}
    if result["status"] == "ok":
        row.update(result["validation_at_0_5"])
        row["fit_seconds"] = result["fit_seconds"]
    else:
        row["error"] = result.get("error", "")
    validation_rows.append(row)

models_validation = pd.DataFrame(validation_rows)
display(models_validation)


In [ ]:
selected = pd.DataFrame([{
    "selected_model": artifact["selected_model"],
    "threshold": float(artifact["threshold"]),
    "selection_metric": "validation average_precision",
    "threshold_objective": "validation F1",
    "validation_AP": metrics["validation"]["average_precision"],
    "validation_AP_above_dummy": metrics["metadata"]["model_selection"]["validation_ap_above_dummy"],
    "validation_F1_at_selected_threshold": metrics["validation"]["f1"],
}])
display(selected)


## 7. 最後才看 test，記錄保留資料的表現

Test 不參與補值、編碼、模型選擇或門檻選擇。下面讀取凍結 Pipeline 在 test 的一次評估結果，並與全猜準時的固定 baseline 比較，不使用 test 結果反覆調參。

- **Precision**：判成延遲的訂單中，實際延遲的比例。
- **Recall**：實際延遲的訂單中，被模型抓到的比例。
- **F1**：precision 與 recall 的調和平均。
- **AP／ROC AUC**：衡量連續分數的排序能力。
- **Accuracy／balanced accuracy**：作為補充，需一起考慮延遲與準時的比例。


In [ ]:
test_keys = [
    "rows", "positive_rate", "threshold", "average_precision", "roc_auc",
    "f1", "recall", "precision", "balanced_accuracy", "accuracy",
]
testmetrics = pd.DataFrame([
    {"model": artifact["selected_model"],
     **{key: metrics["test"][key] for key in test_keys}},
    {"model": "all_on_time_baseline",
     **{key: metrics["test_baseline"][key] for key in test_keys}},
])
display(testmetrics)

confusion = pd.DataFrame(
    metrics["test"]["confusion_matrix"],
    index=["實際準時", "實際延遲"],
    columns=["預測準時", "預測延遲"],
)
display(confusion)
display(Image(filename=str(
    PROJECT_ROOT / "reports" / "images" / "stage4_test_confusion_matrix.png"
)))


## 8. 看哪些特徵影響 validation 的排序

Permutation importance 會打亂某一欄，再看 validation AP 下降多少。下降越多，代表被選模型較依賴該欄；負值或接近零代表這次抽樣沒有顯示穩定的正面貢獻。

圖表使用 validation 抽樣，未用 test 來挑特徵。這是模型行為的描述，不能當成延遲的因果證據；高度相關的欄位也可能分攤重要性。


In [ ]:
display(Image(filename=str(
    PROJECT_ROOT / "reports" / "images" / "stage4_validation_permutation_importance.png"
)))


## 9. 用一筆資料示範 Pipeline 推論

這裡取歷史資料的一筆來示範介面，只把白名單特徵交給儲存的 Pipeline。前處理會沿用訓練時學到的設定；決策門檻沿用 artifact 內 validation 選出的值。

`predict_proba` 輸出作為延遲分數使用。這次沒有做機率校準，分數不應直接解讀成已校準的真實延遲機率。這一筆示範也不構成額外的效能評估。


In [ ]:
example = dataset.iloc[[0]]
feature_row = example.loc[:, artifact["feature_names"]].copy()
pipeline = artifact["pipeline"]
positive_class_index = list(pipeline.classes_).index(1)
delay_score = float(pipeline.predict_proba(feature_row)[0, positive_class_index])
threshold = float(artifact["threshold"])

display(pd.DataFrame([{
    "order_id": example["order_id"].iloc[0],
    "model": artifact["selected_model"],
    "delay_score_uncalibrated": delay_score,
    "threshold": threshold,
    "predicted_delayed": int(delay_score >= threshold),
}]))


## 10. 可選：比較更多模型的開發區間時間回測

新的比較加入 Extra Trees、HistGradient Boosting、XGBoost、LightGBM，連同 Dummy、Logistic Regression 與 Random Forest，一共比較七種模型。報告放在 `reports/stage4/model_comparison.json`，這個區塊只讀取既有成果，缺檔時會顯示執行指令。

這份報告使用三個**開發區間回測**，各自建立 train、validation、evaluation。Validation 與 evaluation 之間留一個月等待區間，讓 validation 的送達標記成熟；等待區間新核准的訂單不加入 validation 或 evaluation。Train 的標記必須在 validation 開始前已知，validation 的標記則必須在 evaluation 開始前已知。各模型在 validation 選門檻後，才評估該 fold 的 evaluation；Dummy 保持固定 0.5 門檻作為參考。

回測的 evaluation 只包含原始固定 test 開始時間（2018-05-26）之前核准的訂單，但保留這些訂單事後才取得的送達答案。因此它是事後整理的歷史評估，並非 evaluation 月底當時就能知道全部結果。

**原先 Random Forest 的固定 test 結果與模型 artifact 保持原樣。** 回測平均 AP 比較反映開發區間的表現，與前面固定 test 的單次評估不同，不能把兩者的指標直接當成同一批訂單的結果。


In [ ]:
COMPARISON_PATH = PROJECT_ROOT / "reports" / "stage4" / "model_comparison.json"
COMPARISON_IMAGE = (
    PROJECT_ROOT / "reports" / "images" / "stage4_backtest_model_comparison.png"
)

if COMPARISON_PATH.is_file():
    comparison_report = json.loads(COMPARISON_PATH.read_text(encoding="utf-8"))
    print("開發區間時間回測報告：", comparison_report["status"])
    comparison_table = pd.json_normalize(comparison_report["summary"])
    comparison_columns = {
        "display_name": "model",
        "status": "status",
        "successful_folds": "successful_folds",
        "development_ap_rank": "development_AP_rank",
        "metrics.average_precision.mean": "AP_mean",
        "metrics.average_precision.std": "AP_std",
        "metrics.roc_auc.mean": "ROC_AUC_mean",
        "metrics.f1.mean": "F1_mean",
        "metrics.recall.mean": "recall_mean",
        "metrics.precision.mean": "precision_mean",
        "metrics.ap_lift_over_prevalence.mean": "AP_lift_over_prevalence_mean",
    }
    display(comparison_table.reindex(columns=comparison_columns).rename(
        columns=comparison_columns
    ))
    if COMPARISON_IMAGE.is_file():
        display(Image(filename=str(COMPARISON_IMAGE)))
else:
    compare_command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "compare_delay_models.py"),
    ]
    print("尚未產生額外模型比較；如要執行，請在終端機使用：")
    print(subprocess.list2cmdline(compare_command))


## 11. 減少誤報：平衡模式與歷史特徵實驗

使用者選定的目標是：validation 至少抓到 30% 延遲訂單，再盡量減少提醒量。新版比較 RF／Logistic、原 21 欄／加入 10 個過去 90 天已送達特徵，所有選模與門檻選擇都只用 validation。

歷史窗口為 `[核准時間−90天, 核准時間)`，答案必須當時已知，同 timestamp 或未來結果不能使用。結果會隨已送達事件更新，但模型不在 evaluation 重訓。

本次歷史特徵沒有勝出，最終平衡 artifact 仍為 static RF。逐月自動選模雖降低合計誤報，卻有一個月召回僅 1.12%，不能只看合計。因此保留原版預設；要使用新門檻，執行 `predict_delay.py --profile balanced`。

下方合計 precision／recall 由加總 TP／FP／FN 計算，與前面每月指標的平均不同。最終保留期間已檢視過，新結果僅是回溯診斷，不是新的獨立測試。缺檔時只顯示指令，不自動重新訓練。

In [ ]:
ALERT_PATH = PROJECT_ROOT / "reports" / "stage4" / "alert_optimization.json"
if ALERT_PATH.is_file():
    alert_report = json.loads(ALERT_PATH.read_text(encoding="utf-8"))
    print("Validation 最低召回目標：", alert_report["min_validation_recall"])
    display(pd.DataFrame(alert_report["summary"])[[
        "model", "policy", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "false_positive_reduction_vs_rf_f1",
    ]])
    display(pd.DataFrame([{
        "month": fold["name"],
        "selected_model": fold["experiment"]["selected_model"],
        "evaluation_recall": fold["experiment"]["models"][fold["experiment"]["selected_model"]]["evaluation"]["balanced"]["recall"],
    } for fold in alert_report["folds"]]))
    final_experiment = alert_report["final"]["experiment"]
    final_name = final_experiment["selected_model"]
    final_model = final_experiment["models"][final_name]
    print("最終候選：", final_name, "門檻：", final_model["thresholds"]["balanced"])
    print("以下為已檢視保留資料的回溯診斷，非新的獨立測試。")
    display(pd.DataFrame([{
        "profile": label, "FP": metrics["confusion_matrix"][0][1],
        "TP": metrics["confusion_matrix"][1][1], "FN": metrics["confusion_matrix"][1][0],
        "precision": metrics["precision"], "recall": metrics["recall"],
    } for label, metrics in (
        ("original", final_experiment["models"]["random_forest_static"]["evaluation"]["f1"]),
        ("balanced", final_model["evaluation"]["balanced"]),
    )]))
    alert_image = PROJECT_ROOT / "reports" / "images" / "stage4_alert_tradeoff.png"
    if alert_image.is_file():
        display(Image(filename=str(alert_image)))
else:
    print("請先執行：", subprocess.list2cmdline([
        sys.executable, str(PROJECT_ROOT / "scripts" / "improve_delay_alerts.py"),
        "--min-recall", "0.30", "--n-jobs", "4",
    ]))


## 12. 可選：月份落差與誤報診斷

這裡讀取 `reports/stage4/monthly_error_diagnostics.json`，回溯檢查已看過的 holdout。診斷使用儲存的平衡模型與**固定決策門檻**，各月都沿用同一門檻；沒有重新訓練、依月份調門檻，或改善模型。這些結果屬於既有保留資料的事後分析，不是新的獨立測試。

月表列出誤報 FP、抓到的延遲 TP、漏掉的延遲 FN 與召回率。**FPR 的分母是實際準時訂單數**，不是提醒數；`partial_month = True` 表示只涵蓋部分月份，不能直接拿件數和完整月比較。

路線表先保留 `sufficient_on_time_rows` 為真的群組，再依 FP 件數取前五名，同時列出準時訂單數與提醒數。即使通過最低樣本量篩選，也要留意分母與小樣本波動。承諾天數使用核准到預估送達的曆日差，實際天數使用核准到送達的連續天數，兩者是描述統計，不能直接替代曆日延遲標記。

這些分組與月份差異描述的是關聯，不能據此判定州別、路線或交期造成誤報。報告缺少時，下方只會顯示產生診斷的指令。


In [ ]:
DIAGNOSTICS_PATH = (
    PROJECT_ROOT / "reports" / "stage4" / "monthly_error_diagnostics.json"
)
if DIAGNOSTICS_PATH.is_file():
    diagnostic_report = json.loads(DIAGNOSTICS_PATH.read_text(encoding="utf-8"))
    print("回溯診斷模型：", diagnostic_report["model"]["selected_model"])
    print("固定門檻：", diagnostic_report["model"]["threshold"])

    month_columns = [
        "month", "rows", "on_time_rows", "false_positives", "false_positive_rate",
        "true_positives", "false_negatives", "recall", "partial_month",
    ]
    monthly_errors = pd.DataFrame(diagnostic_report["test_months"])
    display(monthly_errors[month_columns].rename(columns={
        "false_positives": "FP", "false_positive_rate": "FPR",
        "true_positives": "TP", "false_negatives": "FN",
    }))

    print("準時樣本數足夠的路線：依 FP 件數列前五名")
    route_table = pd.DataFrame(diagnostic_report["groups"]["state_route"])
    if route_table.empty:
        print("沒有可列出的路線群組。")
    else:
        route_top5 = route_table.loc[
            route_table["sufficient_on_time_rows"].eq(True)
        ].sort_values(
            ["false_positives", "false_positive_rate"], ascending=False
        ).head(5)
        display(route_top5[[
            "state_route", "rows", "on_time_rows", "alert_rows",
            "false_positives", "false_positive_rate", "precision", "recall",
        ]].rename(columns={"false_positives": "FP", "false_positive_rate": "FPR"}))

    print("各月承諾與實際送達天數：")
    delivery_days_summary = pd.DataFrame([{
        "month": row["month"],
        "rows": row["rows"],
        "promise_days_mean": row["feature_and_outcome_summary"]["promise_days_mean"],
        "actual_delivery_days_mean": row["feature_and_outcome_summary"]["actual_delivery_days_mean"],
        "partial_month": row["partial_month"],
    } for row in diagnostic_report["test_months"]])
    display(delivery_days_summary)

    for image_name in (
        "stage4_monthly_errors.png", "stage4_false_positive_groups.png"
    ):
        diagnostic_image = PROJECT_ROOT / "reports" / "images" / image_name
        if diagnostic_image.is_file():
            display(Image(filename=str(diagnostic_image)))
else:
    print("尚未產生月份診斷；請在終端機執行：")
    print(subprocess.list2cmdline([
        str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe"),
        str(PROJECT_ROOT / "scripts" / "diagnose_delay_errors.py"),
    ]))


## 13. 可選：等待期與路線承諾特徵實驗

這裡只讀取 `reports/stage4/waiting_strategy_comparison.json` 與既有圖表，不會啟動訓練。**60 天是看結果前指定的主要等待期，30 天是敏感度分析**，不是看 evaluation 成績後挑較好的等待期。每個等待期比較 Random Forest／Logistic Regression、原始特徵／路線承諾特徵，另列 validation 自動選模策略，共 10 種策略。

以 evaluation 開始日 E 為例，validation 核准區間為 `[E−等待天數−30天, E−等待天數)`；train 的核准截止日再比 validation 開始早一個等待期。Train 標籤必須嚴格早於 validation 開始，validation 標籤必須嚴格早於 E。**等待期改變時，train 與 validation 日期、訓練量及資料新近程度也一起改變**，因此比較結果不能解釋成單純延長等待期的因果效果。

路線特徵以主賣家州別 → 顧客州別區分方向；90 天歷史只用核准當時已知的有效送達事件。送達天數分位數與承諾天數都採**曆日尺度**。歷史會隨時間經過及送達結果取得而動態更新，但各折模型不在 evaluation 重新訓練。有效已送達 cohort 仍不包含所有未送達、取消或其他無效訂單，不能把成熟標籤保留率當成所有核准訂單的結果涵蓋率。

所有模型與門檻只用 train／validation 決定；**30% recall 是 validation 的最低目標，不保證後續月份達標**。下方逐月列出 validation 延遲正例保留率與被排除正例數，合計 precision／recall 則由加總 TP／FP／FN 計算，並另列最差月份 recall，避免只看合計。

這三個開發月份已在先前實驗檢視過，屬於開發回測，不是新的未知 test。**本次實驗不評估 2018-05-26 起的原 holdout**。研究 artifact 另存於 `models/research/`，未替換原版或平衡模式的推論預設。


In [ ]:
WAITING_PATH = (
    PROJECT_ROOT / "reports" / "stage4" / "waiting_strategy_comparison.json"
)
if WAITING_PATH.is_file():
    waiting_report = json.loads(WAITING_PATH.read_text(encoding="utf-8"))
    print("預先指定的主要等待期（天）：", waiting_report["primary_wait_days_prespecified"])
    print("Validation 最低召回目標：", waiting_report["min_validation_recall"])
    print("以下為已檢視開發月份的回測；本次不評估原 holdout，也不替換推論預設。")

    waiting_summary = pd.DataFrame(waiting_report["summary"])
    display(waiting_summary[[
        "wait_days", "model", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "minimum_month_recall",
    ]].rename(columns={
        "false_positives": "FP", "true_positives": "TP", "false_negatives": "FN",
    }))

    waiting_fold_rows = []
    for fold in waiting_report["folds"]:
        split = fold["split"]
        validation = split["partitions"]["validation"]
        waiting_fold_rows.append({
            "wait_days": fold["wait_days"],
            "month": fold["name"],
            "selected_model": fold["experiment"]["selected_model"],
            "train_prediction_end": split["train_prediction_end"],
            "validation_start": split["validation_start"],
            "validation_end": split["validation_end"],
            "validation_positive_rows_before_purge": validation["positive_rows_before_purge"],
            "validation_positive_retention_rate": validation["positive_retention_rate"],
            "validation_purged_positive_rows": validation["purged_positive_rows"],
        })
    print("有效已送達 cohort 內的標籤成熟情況；非所有核准訂單涵蓋率：")
    display(pd.DataFrame(waiting_fold_rows))

    waiting_image = PROJECT_ROOT / "reports" / "images" / "stage4_waiting_tradeoff.png"
    if waiting_image.is_file():
        display(Image(filename=str(waiting_image)))
else:
    print("python scripts/compare_waiting_strategies.py --n-jobs 4")


## 本階段留下的成果

- 可重用的訂單特徵與延遲標記建置程式。
- 使用時間切分與成熟標記 purge 的模型比較。
- 固定模型、前處理與決策門檻的本機 artifact。
- Validation 選模結果、test 指標、混淆矩陣與特徵重要性圖。
- 可用同一套 Pipeline 進行推論的介面。

下一步可依專案目標選擇：把這個模型接成 API、討論延遲判斷的商業成本與門檻，或繼續 Stage 5 的評論好壞預測。這些工作需要各自的資料與評估設計。
